In [1]:
#!/usr/bin/env python3
"""
Generate all parameter .in files for the Z4 domain-wall model scan.

Physics:
  V = -(μ²/2)(h²+a²) + (λ₁/4)(h²+a²)² − (λ₂μ/2)(h⁴−6h²a²+a⁴) + V₀

  dV/dh|vac = 0  →  v² = μ² / (λ₁ − 2μλ₂)
  Vacuum vev:  v = μ / sqrt(λ₁ − 2μλ₂)
  Unit-vev constraint (v=1):  λ₁ = μ² + 2μλ₂

Sweep axes:
  mu    : [0.02, 0.05, 0.10, 0.5, 1.0]     – physical mass scale; H0 = H0_RATIO * mu
  beta  : [0.01, 0.05, 0.10, 0.40, 1, 2]   – β = 3λ₂/sqrt(8λ₁) > 0

  Derived (from v=1 and β = 3λ₂/sqrt(8λ₁)):
    c      = β*sqrt(8)/3
    x      = sqrt(λ₁) = μ * (c + sqrt(1 + c²))
    λ₁     = x²
    λ₂     = c*x

Fixed:
  H0      = 1.0 * mu   (change H0_RATIO to 0.3 for a second pass)
  kCutOff = [1, 3, 5]
  N = 512, lSide = 100, dt = 0.01, tMax = 50
"""

import os
import numpy as np

# ── Sweep axes ────────────────────────────────────────────────────────────────
MU_VALS   = [ 0.05, 0.10]      # physical mass scale
BETA_VALS = [0.001,0.2, 0.25, 1,5,7]     # beta = 3*lambda2/sqrt(8*lambda1)
KCUTOFFS  = [ 3]

# ── Fixed parameters ──────────────────────────────────────────────────────────
H0_RATIO = 1.0      # H0 = H0_RATIO * mu  (try 0.3 for a second scan)

N        = 512
LSIDE    = 100
DT       = 0.01
T_MAX    = 90
REMOTE_BASE = "/mt/user-batch/dpasari/scan_z4_targeted"

# ── Helpers ───────────────────────────────────────────────────────────────────
def float_str(x: float) -> str:
    """Convert float to a filename-safe string: 0.05 -> '0p05', 0.1 -> '0p1'."""
    return f"{x:g}".replace(".", "p")

def h0_label(ratio: float) -> str:
    """0.3 -> 'H0p3mu',  1.0 -> 'H1mu'."""
    r = f"{ratio:g}".replace(".", "p")
    return f"H{r}mu"

def couplings_from_mu_beta(mu: float, beta: float):
    """Solve {v=1, beta=3*lambda2/sqrt(8*lambda1)} for (lambda1, lambda2)."""
    assert beta > 0.0, f"Need beta > 0, got {beta}"
    c = beta * np.sqrt(8.0) / 3.0
    x = mu * (c + np.sqrt(1.0 + c**2))   # x = sqrt(lambda1) > 0
    lambda1 = x**2
    lambda2 = c * x
    assert abs(lambda1) < 16*3.14, f"got lambda1 too big, got {lambda1}"
    assert abs(lambda2*mu) < 4*3.14, f"got lambda2 too big, got {lambda1}"
    return lambda1, lambda2

TEMPLATE = """\
#Output
outputfile = {results_dir}

#Evolution
expansion = true
evolver = LF

#Lattice
N = {N}
dt = {dt}
lSide = {lSide}

#Times
t0 = 0
tOutputFreq  = 0.1
tOutputInfreq  = 5
tOutputRareFreq = 3
tMax = {tmax}

#Spectra options
PS_type = 1
PS_version = 1

#GWs
GWprojectorType = 2
withGWs = false

fixedBackground = true
omegaEoS = 0.3333
H0 = {H0:.8f}   # {H0_ratio:.4f} * mu = {H0_ratio:.4f} * {mu}

#IC
kCutOff = {kcut}
initial_amplitudes = 0 0
initial_momenta    = 0 0

# Model Parameters  (Z4 domain-wall model)
# Vacuum vev:  v = mu / sqrt(lambda1 - 2*mu*lambda2) = 1  (unit vev)
# Constraint:  lambda1 = mu^2 + 2*mu*lambda2
# Z4 strength: beta = 3*lambda2/sqrt(8*lambda1) = {beta:.4f}  (>0)
# Check:  lambda1 - 2*mu*lambda2 = mu^2 = {mu2:.2e}  (must equal mu^2 for v=1)
mu      = {mu}
lambda1 = {lambda1:.10f}
lambda2 = {lambda2:.10f}
"""

# ── Resolve output directory relative to notebook location ────────────────────
try:
    _root = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _root = os.getcwd()  # Jupyter: cwd should be the analysis/ folder

out_dir = os.path.join(_root, "..", "src", "models", "parameter-files", "scan_z4_targeted")
os.makedirs(out_dir, exist_ok=True)

# ── Generate files ────────────────────────────────────────────────────────────
generated = []
print(f"{'File':<60} {'mu':>7} {'beta':>7} {'kcut':>6} {'lambda1':>16} {'lambda2':>16} {'H0':>10}")
print("-" * 130)

for mu in MU_VALS:
    for beta in BETA_VALS:
        lambda1, lambda2 = couplings_from_mu_beta(mu, beta)
        H0 = H0_RATIO * mu

        # Sanity checks
        beta_back = 3.0 * lambda2 / np.sqrt(8.0 * lambda1)
        assert abs(beta_back - beta) < 1e-12, f"beta mismatch: {beta_back} vs {beta}"
        vev = mu / np.sqrt(lambda1 - 2.0 * mu * lambda2)
        assert abs(vev - 1.0) < 1e-10, f"vev = {vev} != 1 for mu={mu}, beta={beta}"

        mu_s = float_str(mu)
        beta_s = float_str(beta)
        hl = h0_label(H0_RATIO)

        for kcut in KCUTOFFS:
            fname   = f"DWZ4_mu{mu_s}_beta{beta_s}_{hl}_k{kcut}.in"
            res_dir = f"{REMOTE_BASE}/results_z4_mu{mu_s}_beta{beta_s}_{hl}_k{kcut}/"

            content = TEMPLATE.format(
                results_dir = res_dir,
                N           = N,
                dt          = DT,
                lSide       = LSIDE,
                tmax        = T_MAX,
                H0          = H0,
                H0_ratio    = H0_RATIO,
                kcut        = kcut,
                mu          = mu,
                mu2         = mu**2,
                lambda1     = lambda1,
                lambda2     = lambda2,
                beta        = beta,
            )

            fpath = os.path.join(out_dir, fname)
            with open(fpath, "w") as fh:
                fh.write(content)
            generated.append(fname)

            print(f"{fname:<60} {mu:7g} {beta:7g} {kcut:6d} {lambda1:16.6e} {lambda2:16.6e} {H0:10.4e}")

print("-" * 130)
print(f"Generated {len(generated)} files → {os.path.abspath(out_dir)}")


File                                                              mu    beta   kcut          lambda1          lambda2         H0
----------------------------------------------------------------------------------------------------------------------------------
DWZ4_mu0p05_beta0p001_H1mu_k3.in                                0.05   0.001      3     2.504718e-03     4.718492e-05 5.0000e-02
DWZ4_mu0p05_beta0p2_H1mu_k3.in                                  0.05     0.2      3     3.637201e-03     1.137201e-02 5.0000e-02
DWZ4_mu0p05_beta0p25_H1mu_k3.in                                 0.05    0.25      3     3.988583e-03     1.488583e-02 5.0000e-02
DWZ4_mu0p05_beta1_H1mu_k3.in                                    0.05       1      3     1.342328e-02     1.092328e-01 5.0000e-02
DWZ4_mu0p05_beta5_H1mu_k3.in                                    0.05       5      3     2.271947e-01     2.246947e+00 5.0000e-02
DWZ4_mu0p05_beta7_H1mu_k3.in                                    0.05       7      3     4.40541

In [2]:
# ── Quick summary table ───────────────────────────────────────────────────────
# v = mu / sqrt(lambda1 - 2*mu*lambda2) should be 1.0 for every row.
# beta = 3*lambda2/sqrt(8*lambda1) is the Z4-strength scan coordinate.

import pandas as pd

rows = []
for mu in MU_VALS:
    for beta in BETA_VALS:
        lambda1, lambda2 = couplings_from_mu_beta(mu, beta)
        H0 = H0_RATIO * mu
        vev = mu / (lambda1 - 2.0 * mu * lambda2)**0.5
        rows.append(dict(
            mu=mu, beta=beta,
            lambda1=round(lambda1, 8),
            lambda2=round(lambda2, 8),
            H0=round(H0, 6),
            vev=round(vev, 8),
        ))

df = pd.DataFrame(rows)
print(df.to_string(index=False))


ModuleNotFoundError: No module named 'pandas'